In [1]:
# MODELO 3 - STEMMING TFIDF

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import ast
import re

from google.colab import files

from nltk.stem import PorterStemmer
import nltk
nltk.download('punkt')

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

# CARGA DEL ARCHIVO

dataTraining = pd.read_csv(
    "https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip",
    encoding="UTF-8",
    index_col=0
)

dataTesting = pd.read_csv(
    "https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip",
    encoding="UTF-8",
    index_col=0
)

# STEMMING

stemmer = PorterStemmer()

def clean_and_stem(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    words = text.split()
    words = [stemmer.stem(word) for word in words]
    return " ".join(words)

dataTraining["genres"] = dataTraining["genres"].apply(ast.literal_eval)

# título duplicado
X = (
    dataTraining["title"].fillna("") + " " +
    dataTraining["title"].fillna("") + " " +
    dataTraining["plot"].fillna("")
).apply(clean_and_stem)

X_test = (
    dataTesting["title"].fillna("") + " " +
    dataTesting["title"].fillna("") + " " +
    dataTesting["plot"].fillna("")
).apply(clean_and_stem)


# TARGET

le = MultiLabelBinarizer()
y = le.fit_transform(dataTraining["genres"])


# MODELO


model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        strip_accents="unicode",
        sublinear_tf=True,
        ngram_range=(1,3),
        max_features=40000,
        min_df=1,
        max_df=0.9
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",
            C=1.15,
            max_iter=3000,
            random_state=42
        ),
        n_jobs=-1
    ))
])

print("Entrenando modelo con stemming...")
model.fit(X, y)


# PREDICCIÓN

pred_test = model.predict_proba(X_test)

cols = [
    "p_Action", "p_Adventure", "p_Animation", "p_Biography", "p_Comedy",
    "p_Crime", "p_Documentary", "p_Drama", "p_Family", "p_Fantasy",
    "p_Film-Noir", "p_History", "p_Horror", "p_Music", "p_Musical",
    "p_Mystery", "p_News", "p_Romance", "p_Sci-Fi", "p_Short",
    "p_Sport", "p_Thriller", "p_War", "p_Western"
]

res = pd.DataFrame(
    pred_test,
    index=dataTesting.index,
    columns=cols
)

filename = "pred_stemming_tfidf.csv"

res.to_csv(filename, index_label="ID")

files.download(filename)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Entrenando modelo con stemming...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>